In [1]:
!pip install openai langchain_core langchain_openai

import os
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings  # langchain.embeddings 경로는 deprecated

embeddings = OpenAIEmbeddings(
    model="embedding-8b:sl",
    base_url="http://host.docker.internal:12345/v1",   # /v1 까지 붙여야 함
    api_key="lm-studio",                       # LM Studio는 키 검증 안 함 → 더미 값
    check_embedding_ctx_length=False,          # ★ 토큰ID 대신 원문 문자열 전송
)


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
query_result = embeddings.embed_query('저는 배가 고파요')
print(len(query_result))   # 1536(ada-002) 아님. 모델 차원 그대로 (8B 계열이면 보통 4096)
print(query_result[:5])

4096
[0.015875395387411118, 0.009350869804620743, -0.0034897313453257084, -0.04658934846520424, 0.011201159097254276]


In [3]:
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]
df = pd.DataFrame(data, columns=['text'])

In [4]:
def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row: get_embedding(row.text), axis=1)
df

,text,embedding
0,주식 시장이 급등했어요,"[0.005533280316740274, 0.009821378625929356, -..."
1,시장 물가가 올랐어요,"[0.014084487222135067, 0.01855550706386566, -0..."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.010446210391819477, 0.020618287846446037, ..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[0.010052042081952095, 0.008625758811831474, -..."
4,저는 빠른 비트를 좋아해요,"[0.029428981244564056, 2.928206777141895e-05, ..."
5,최근 비트코인 가격이 많이 변동했어요,"[0.026050791144371033, 0.018739107996225357, 0..."


In [5]:
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

def return_answer_candidate(df, query):
    query_embedding = get_embedding(query)
    df["similarity"] = df.embedding.apply(
        lambda x: cos_sim(np.array(x), np.array(query_embedding))
    )
    return df.sort_values("similarity", ascending=False).head(3)

sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.014084487222135067, 0.01855550706386566, -0...",0.711535
0,주식 시장이 급등했어요,"[0.005533280316740274, 0.009821378625929356, -...",0.592786
4,저는 빠른 비트를 좋아해요,"[0.029428981244564056, 2.928206777141895e-05, ...",0.575370


In [6]:
def get_detailed_instruct(task: str, query: str) -> str:
    return f'Instruct: {task}\nQuery:{query}'        # 공식 포맷 (Query: 뒤 공백 없음)

def get_embedding_instruct(text: str, task: str):    # get_embedding의 인스트럭션 버전
    return embeddings.embed_query(get_detailed_instruct(task, text))

def return_answer_candidate_instruct(df, query, task):
    query_embedding = get_embedding_instruct(query, task)   # 쿼리만 인스트럭션
    sim = df.embedding.apply(                                # 문서는 기존 무인스트럭션 벡터 재사용
        lambda x: cos_sim(np.array(x), np.array(query_embedding))
    )
    out = df.assign(similarity=sim)                          # df 원본 비오염 (비교 실험용)
    return out.sort_values("similarity", ascending=False).head(3)

In [7]:
query = '과일 값이 비싸다'

print('=== baseline (no instruction) ===')
print(return_answer_candidate(df, query)[['text', 'similarity']].to_string(), '\n')

tasks = {
    'A_generic'  : 'Given a query, retrieve sentences with semantically similar meaning',
    'B_price'    : 'Given a statement about rising prices, retrieve sentences about price increases or cost of living',
    'C_websearch': 'Given a web search query, retrieve relevant passages that answer the query',
}

for name, task in tasks.items():
    print(f'=== {name} ===')
    print(return_answer_candidate_instruct(df, query, task)[['text', 'similarity']].to_string(), '\n')

=== baseline (no instruction) ===
             text  similarity
1     시장 물가가 올랐어요    0.711535
0    주식 시장이 급등했어요    0.592786
4  저는 빠른 비트를 좋아해요    0.575370 

=== A_generic ===
                   text  similarity
1           시장 물가가 올랐어요    0.718660
0          주식 시장이 급등했어요    0.595801
5  최근 비트코인 가격이 많이 변동했어요    0.582413 

=== B_price ===
                   text  similarity
1           시장 물가가 올랐어요    0.579723
5  최근 비트코인 가격이 많이 변동했어요    0.469305
0          주식 시장이 급등했어요    0.445056 

=== C_websearch ===
                   text  similarity
1           시장 물가가 올랐어요    0.387476
5  최근 비트코인 가격이 많이 변동했어요    0.357800
0          주식 시장이 급등했어요    0.296060 



In [8]:
commerce_tasks = {
    'D_place': 'Given a query, retrieve sentences about marketplaces or places where goods are sold',
    'E_buy'  : 'Given a query about buying a product, retrieve sentences about where to buy or sell goods',
}

# TEST 1 (순수 인스트럭션 효과): 쿼리 고정 + 상거래 인스트럭션
#   → 쿼리 자체가 "비싸다(가격)"라 가격 편향과 싸우는 어려운 테스트
print('### TEST 1: query="과일 값이 비싸다" ###')
for name, task in commerce_tasks.items():
    print(f'--- {name} ---')
    print(return_answer_candidate_instruct(df, '과일 값이 비싸다', task)[['text','similarity']].to_string(), '\n')

# TEST 2 (결정적 테스트): 장소/구매 의도 쿼리 + 상거래 인스트럭션
print('### TEST 2: query="과일을 어디서 살 수 있나요" ###')
for name, task in commerce_tasks.items():
    print(f'--- {name} ---')
    print(return_answer_candidate_instruct(df, '과일을 어디서 살 수 있나요', task)[['text','similarity']].to_string(), '\n')


### TEST 1: query="과일 값이 비싸다" ###
--- D_place ---
                   text  similarity
1           시장 물가가 올랐어요     0.51936
2  전통 시장에는 다양한 물품들을 팔아요     0.46762
0          주식 시장이 급등했어요     0.39937 

--- E_buy ---
                   text  similarity
1           시장 물가가 올랐어요    0.429578
2  전통 시장에는 다양한 물품들을 팔아요    0.410015
0          주식 시장이 급등했어요    0.314971 

### TEST 2: query="과일을 어디서 살 수 있나요" ###
--- D_place ---
                   text  similarity
2  전통 시장에는 다양한 물품들을 팔아요    0.507305
1           시장 물가가 올랐어요    0.363090
0          주식 시장이 급등했어요    0.291071 

--- E_buy ---
                   text  similarity
2  전통 시장에는 다양한 물품들을 팔아요    0.457021
1           시장 물가가 올랐어요    0.332161
0          주식 시장이 급등했어요    0.260202 

